# 📊 Analyse des Réglements par CAM
Upload votre fichier `.txt` mensuel, puis exécutez les cellules.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import re
from collections import defaultdict
import pandas as pd
from google.colab import files
import io

pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# ── Configuration ───────────────────────────────────────────────────
TARGET_CAMS = {
    'CAM01','CAM02','CAM03','CAM04','CAM05','CAM06',
    'CAM36','CAM37','CAM38','CAM48','CAM49'
}

TYPE_MAP = {
    'CESP':  'Espèces',
    'CTRT':  'Traite',
    'CCHQR': 'Chèque',
}

print('✅ Configuration OK')

In [ ]:
# ── Upload du fichier ────────────────────────────────────────────────
uploaded = files.upload()
filename = list(uploaded.keys())[0]
try:
    text = uploaded[filename].decode('utf-8')
except UnicodeDecodeError:
    text = uploaded[filename].decode('latin-1')

print(f'📄 Fichier chargé : {filename} ({len(text.splitlines())} lignes)')

In [ ]:
# ── Parsing ──────────────────────────────────────────────────────────
def parse_lines(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split(';')
        if len(parts) < 11:
            continue
        code = parts[0]
        ref2 = parts[1]
        try:
            amount = float(parts[-1])
        except ValueError:
            continue
        prefix = next((p for p in TYPE_MAP if code.startswith(p)), None)
        if prefix is None:
            continue
        cam_match = re.search(r'(CAM\d+)', ref2)
        cam = cam_match.group(1) if cam_match else None
        rows.append({
            'Code'         : code,
            'CAM'          : cam,
            'Type'         : TYPE_MAP[prefix],
            'Montant (TND)': amount,
        })
    return pd.DataFrame(rows)

df = parse_lines(text)
df_cam = df[df['CAM'].isin(TARGET_CAMS)].copy()

print(f'Total lignes parsées     : {len(df)}')
print(f'Lignes CAMs ciblées      : {len(df_cam)}')
print(f'Lignes hors périmètre    : {len(df) - len(df_cam)}')
df_cam.head()

In [ ]:
# ── Résumé par type de règlement ─────────────────────────────────────
print('\n=== RÉSUMÉ PAR TYPE ===')
type_summary = (df_cam.groupby('Type')['Montant (TND)']
                .agg(Total='sum', Transactions='count')
                .sort_values('Total', ascending=False))
type_summary['% du total'] = (type_summary['Total'] / type_summary['Total'].sum() * 100).round(2)
print(type_summary.to_string())

In [ ]:
# ── Classement des CAMs (total + détail par type) ────────────────────

# Total par CAM
totals = (df_cam.groupby('CAM')['Montant (TND)']
          .agg(Total='sum', Transactions='count')
          .reset_index()
          .sort_values('Total', ascending=False)
          .reset_index(drop=True))
totals.index += 1
totals.index.name = 'Rang'

# Détail par CAM × Type (pivot)
pivot = (df_cam.groupby(['CAM','Type'])['Montant (TND)']
         .sum()
         .unstack(fill_value=0)
         .round(3))

# Merge
ranking = totals.join(pivot, on='CAM')
grand_total = ranking['Total'].sum()
ranking['Part (%)'] = (ranking['Total'] / grand_total * 100).round(2)

print(f'\n=== CLASSEMENT DES CAMs — TOTAL GÉNÉRAL: {grand_total:,.3f} TND ===')
print(ranking.to_string())

In [ ]:
# ── Visualisation ASCII barchart ─────────────────────────────────────
print('\n=== BARCHART PAR CAM ===')
max_total = ranking['Total'].max()
bar_width  = 40
for _, row in ranking.iterrows():
    bar_len = int((row['Total'] / max_total) * bar_width)
    bar = '█' * bar_len + '░' * (bar_width - bar_len)
    print(f"{str(row['CAM']).ljust(7)} {bar}  {row['Total']:>12,.3f} TND  ({row['Part (%)']:.1f}%)")

In [ ]:
# ── Export CSV ───────────────────────────────────────────────────────
out_csv = ranking.reset_index().to_csv(index=False)
files.download_bytes = lambda name, data: files.download(io.BytesIO(data.encode()).read())

with open('classement_cam.csv', 'w', encoding='utf-8-sig') as f:
    f.write(out_csv)
files.download('classement_cam.csv')
print('✅ CSV téléchargé : classement_cam.csv')